In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

This notebook shows how to run lakehouse with external tables by defining the path function

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("properties", get_properties(F.col("url")))

    def path(self, table: str) -> str:
        return f"D:/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite", external=True).execute(
    "people"
)

In [10]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-02-17 20:30:...|        Cliegg Lars| 62|https://www.swapi...|{"height": "183",...|
|2025-02-17 20:30:...|  Poggle the Lesser| 63|https://www.swapi...|{"height": "183",...|
|2025-02-17 20:30:...|    Luminara Unduli| 64|https://www.swapi...|{"height": "170",...|
|2025-02-17 20:30:...|      Barriss Offee| 65|https://www.swapi...|{"height": "166",...|
|2025-02-17 20:30:...|              Dormé| 66|https://www.swapi...|{"height": "165",...|
|2025-02-17 20:30:...|              Dooku| 67|https://www.swapi...|{"height": "193",...|
|2025-02-17 20:30:...|Bail Prestor Organa| 68|https://www.swapi...|{"height": "191",...|
|2025-02-17 20:30:...|         Jango Fett| 69|https://www.swapi...|{"height": "183",...|
|2025-02

# 2 Silver

In [11]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [13]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            sdf = self.transf_people(sdf)
        return sdf

    def transf_people(self, sdf: DataFrame) -> DataFrame:
        sdf = (
            sdf.withColumn("height", sdf.properties.height)
            .withColumn("mass", sdf.properties.mass)
            .withColumn("gender", sdf.properties.gender)
            .drop("url", "properties")
        )
        for i in range(0, 35):
            sdf = sdf.withColumn(f"col{str(i)}", F.lit(str(i)))
        return sdf

    def path(self, table: str) -> str:
        return f"D:/{self.catalog}/{self.target_schema}/{table}"


silver_instance = StarWarsSilver(spark, **options)

In [14]:
silver_instance.load().transform().write(mode="overwrite", external=True).execute(
    "people"
)

In [15]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 82
+--------------------------+-------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|LH_SilverTS               |LH_BronzeTS              |name                 |uid|height |mass   |gender       |col0|col1|col2|col3|col4|col5|col6|col7|col8|col9|col10|col11|col12|col13|col14|col15|col16|col17|col18|col19|col20|col21|col22|col23|col24|col25|col26|col27|col28|col29|col30|col31|col32|col33|col34|
+--------------------------+-------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|2025-02-17 20:31:21.174401|2025-02-17 20:30:41.11425|

# 4 Clean Up

In [16]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")

DataFrame[]